In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Sklearn para modelado
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, 
    classification_report, 
    roc_curve, 
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Para visualización de métricas
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

# Configuración para ver más columnas
pd.set_option('display.max_columns', 50)

import warnings
warnings.filterwarnings('ignore')

---

## Parte 1: Carga y exploración inicial de datos

In [ ]:
# Cargar el dataset
df = pd.read_csv("https://raw.githubusercontent.com/ami232/nebrija-mpan/main/data/hotel_bookings.csv")

print(f"Dimensiones del dataset: {df.shape}")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())

In [ ]:
# Ver primeras filas
df.head()

In [ ]:
# Información sobre tipos de datos y valores nulos
df.info()

### Variable objetivo: `is_canceled`

Esta es nuestra variable dependiente (target). Indica si la reserva fue cancelada (1) o no (0).

In [ ]:
# Distribución de la variable objetivo
print("Distribución de cancelaciones:")
print(df["is_canceled"].value_counts())
print(f"\nPorcentaje de cancelaciones: {df['is_canceled'].mean()*100:.1f}%")

**Tarea:** Crea un gráfico de barras que muestre la distribución de cancelaciones (con etiquetas de porcentaje).

In [ ]:
# Gráfico de barras de distribución de cancelaciones
fig, ax = plt.subplots(figsize=(8, 5))

# Calcular porcentajes
counts = df["is_canceled"].value_counts()
percentages = df["is_canceled"].value_counts(normalize=True) * 100

# Crear gráfico
bars = ax.bar(
    ["No Cancelado", "Cancelado"], counts.values, color=["#2ecc71", "#e74c3c"]
)

# Añadir etiquetas de porcentaje
for bar, pct in zip(bars, percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500,
        f"{pct:.1f}%",
        ha="center",
        fontsize=12,
    )

ax.set_ylabel("Número de reservas")
ax.set_title("Distribución de Cancelaciones")
plt.show()

### Diccionario de variables principales

| Variable | Descripción | Tipo |
|----------|-------------|------|
| `hotel` | Tipo de hotel (Resort/City) | Categórica |
| `is_canceled` | Si la reserva fue cancelada | Binaria (target) |
| `lead_time` | Días entre reserva y llegada | Numérica |
| `arrival_date_month` | Mes de llegada | Categórica |
| `stays_in_weekend_nights` | Noches de fin de semana | Numérica |
| `stays_in_week_nights` | Noches entre semana | Numérica |
| `adults` | Número de adultos | Numérica |
| `children` | Número de niños | Numérica |
| `babies` | Número de bebés | Numérica |
| `meal` | Tipo de régimen | Categórica |
| `country` | País de origen | Categórica |
| `market_segment` | Segmento de mercado | Categórica |
| `is_repeated_guest` | Si es cliente repetidor | Binaria |
| `previous_cancellations` | Cancelaciones previas | Numérica |
| `deposit_type` | Tipo de depósito | Categórica |
| `customer_type` | Tipo de cliente | Categórica |
| `adr` | Tarifa media diaria (Average Daily Rate) | Numérica |
| `total_of_special_requests` | Peticiones especiales | Numérica |

---

## Parte 2: Análisis exploratorio orientado a la predicción

Queremos entender qué variables están relacionadas con la cancelación.

### 2.1 Análisis de variables numéricas vs cancelación

In [ ]:
# Variables numéricas de interés
vars_numericas = [
    "lead_time",
    "adr",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "previous_cancellations",
    "total_of_special_requests",
]

# Estadísticos por grupo de cancelación
df.groupby("is_canceled")[vars_numericas].mean().round(2)

**Tarea:** Crea boxplots comparando la distribución de `lead_time` y `adr` entre reservas canceladas y no canceladas.

In [ ]:
# Boxplots comparando lead_time y adr entre canceladas y no canceladas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot de lead_time
sns.boxplot(data=df, x="is_canceled", y="lead_time", ax=axes[0])
axes[0].set_xticklabels(["No Cancelado", "Cancelado"])
axes[0].set_xlabel("Estado de la reserva")
axes[0].set_ylabel("Lead Time (días)")
axes[0].set_title("Lead Time por Estado de Cancelación")

# Boxplot de adr
sns.boxplot(data=df, x="is_canceled", y="adr", ax=axes[1])
axes[1].set_xticklabels(["No Cancelado", "Cancelado"])
axes[1].set_xlabel("Estado de la reserva")
axes[1].set_ylabel("ADR (€)")
axes[1].set_title("Tarifa Media Diaria por Estado de Cancelación")

plt.tight_layout()
plt.show()

**Pregunta:** ¿Qué observas sobre el `lead_time` en reservas canceladas vs no canceladas? ¿Tiene sentido desde una perspectiva de negocio?

### 2.2 Análisis de variables categóricas vs cancelación

In [ ]:
# Tasa de cancelación por tipo de hotel
print("Tasa de cancelación por tipo de hotel:")
df.groupby("hotel")["is_canceled"].mean().round(3)

**Tarea:** Calcula y visualiza la tasa de cancelación por:
1. `deposit_type`
2. `customer_type`
3. `market_segment`

In [ ]:
# Tasa de cancelación por deposit_type
print("Tasa de cancelación por tipo de depósito:")
cancel_by_deposit = (
    df.groupby("deposit_type")["is_canceled"].mean().sort_values(ascending=False)
)
print(cancel_by_deposit)

# Visualización
fig, ax = plt.subplots(figsize=(8, 5))
cancel_by_deposit.plot(kind="bar", ax=ax, color="steelblue")
ax.set_ylabel("Tasa de cancelación")
ax.set_xlabel("Tipo de depósito")
ax.set_title("Tasa de Cancelación por Tipo de Depósito")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Tasa de cancelación por customer_type
print("Tasa de cancelación por tipo de cliente:")
cancel_by_customer = (
    df.groupby("customer_type")["is_canceled"].mean().sort_values(ascending=False)
)
print(cancel_by_customer)

# Visualización
fig, ax = plt.subplots(figsize=(8, 5))
cancel_by_customer.plot(kind="bar", ax=ax, color="steelblue")
ax.set_ylabel("Tasa de cancelación")
ax.set_xlabel("Tipo de cliente")
ax.set_title("Tasa de Cancelación por Tipo de Cliente")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Tasa de cancelación por market_segment
print("Tasa de cancelación por segmento de mercado:")
cancel_by_segment = (
    df.groupby("market_segment")["is_canceled"].mean().sort_values(ascending=False)
)
print(cancel_by_segment)

# Visualización
fig, ax = plt.subplots(figsize=(10, 5))
cancel_by_segment.plot(kind="bar", ax=ax, color="steelblue")
ax.set_ylabel("Tasa de cancelación")
ax.set_xlabel("Segmento de mercado")
ax.set_title("Tasa de Cancelación por Segmento de Mercado")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

**Pregunta:** ¿Qué tipo de depósito tiene la mayor tasa de cancelación? ¿Es contraintuitivo? ¿Por qué crees que ocurre esto?

### 2.3 Efecto de ser cliente repetidor

In [ ]:
# Tasa de cancelación por tipo de cliente repetidor
print("Tasa de cancelación según si es cliente repetidor:")
df.groupby("is_repeated_guest")["is_canceled"].agg(["mean", "count"])

---

## Parte 3: Preparación de datos

Antes de construir el modelo, necesitamos:
1. Tratar valores nulos
2. Seleccionar variables
3. Transformar variables categóricas

### 3.1 Tratamiento de valores nulos

In [ ]:
# Verificar valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Crear copia del dataset para trabajar
df_model = df.copy()

# Tratar valores 'NULL' como string en agent y company
df_model["agent"] = df_model["agent"].replace("NULL", np.nan)
df_model["company"] = df_model["company"].replace("NULL", np.nan)

# Rellenar children con 0 (asumimos que si no hay info, no hay niños)
df_model["children"] = df_model["children"].fillna(0)

### 3.2 Selección de variables

Vamos a seleccionar un subconjunto de variables relevantes para nuestro modelo inicial.

In [ ]:
# Variables seleccionadas para el modelo
variables_modelo = [
    # Numéricas
    "lead_time",
    "adr",
    "total_of_special_requests",
    "previous_cancellations",
    # Binarias (ya numéricas)
    "is_repeated_guest",
    # Categóricas (necesitan transformación)
    "hotel",
    "deposit_type",
    "customer_type",
]

# Variable objetivo
target = "is_canceled"

### 3.3 Transformación de variables categóricas

Para usar variables categóricas en regresión logística, necesitamos convertirlas a variables dummy (one-hot encoding).

In [ ]:
# Seleccionar variables
df_selected = df_model[variables_modelo + [target]].copy()

# Eliminar filas con valores nulos en adr (si hay)
df_selected = df_selected.dropna(subset=["adr"])

print(f"Filas después de limpieza: {len(df_selected)}")
df_selected.head()

**Tarea:** Usa `pd.get_dummies()` para crear variables dummy de las variables categóricas (`hotel`, `deposit_type`, `customer_type`). Usa el parámetro `drop_first=True` para evitar multicolinealidad.

In [ ]:
# Identificar variables categóricas
vars_categoricas = ["hotel", "deposit_type", "customer_type"]

# Crear dummies con drop_first=True para evitar multicolinealidad
df_dummies = pd.get_dummies(df_selected, columns=vars_categoricas, drop_first=True)

print(f"Dimensiones después de dummies: {df_dummies.shape}")
df_dummies.head()

In [ ]:
# Verificar las columnas resultantes
print("Columnas después de crear dummies:")
print(df_dummies.columns.tolist())

---

## Parte 4: Construcción del modelo de regresión logística

### 4.1 Entrenar el modelo

**Tarea:** Crea una instancia de `LogisticRegression` y entrénala con los datos.

Usa `max_iter=1000` para asegurar convergencia.

In [ ]:
# Separar features (X) y target (y)
X = df_dummies.drop(columns=[target])
y = df_dummies[target]

# Crear y entrenar el modelo de regresión logística
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

print("Modelo entrenado correctamente")
print(f"\nCoeficientes del modelo:")
for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"  {feature}: {coef:.4f}")
print(f"\nIntercepto: {model.intercept_[0]:.4f}")

### 4.2 Realizar predicciones

In [ ]:
# Predicciones de clase (0 o 1)
y_pred = model.predict(X)

# Predicciones de probabilidad
y_prob = model.predict_proba(X)[:, 1]  # Probabilidad de clase 1 (cancelación)

print("Ejemplo de predicciones:")
print(f"Clases predichas: {y_pred[:10]}")
print(f"Probabilidades: {y_prob[:10].round(3)}")

---

## Parte 5: Evaluación del modelo

### 5.1 Matriz de confusión

In [ ]:
# Calcular matriz de confusión
cm = confusion_matrix(y, y_pred)

# Visualizar
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=["No canceló", "Canceló"]).plot(
    ax=ax, cmap="Blues"
)
plt.title("Matriz de Confusión")
plt.show()

# Mostrar valores
print(f"\nVerdaderos Negativos (TN): {cm[0,0]}")
print(f"Falsos Positivos (FP): {cm[0,1]}")
print(f"Falsos Negativos (FN): {cm[1,0]}")
print(f"Verdaderos Positivos (TP): {cm[1,1]}")

**Pregunta:** En el contexto del hotel:
- ¿Qué significa un **Falso Negativo** (FN)? ¿Cuál es su coste?
- ¿Qué significa un **Falso Positivo** (FP)? ¿Cuál es su coste?

### 5.2 Métricas de clasificación

**Tarea:** Calcula y muestra las siguientes métricas:
- Accuracy
- Precision
- Recall
- F1-Score

Puedes usar `classification_report` o calcularlas individualmente.

In [ ]:
# Calcular métricas de clasificación
print("=== Métricas de Clasificación ===\n")

# Accuracy
acc = accuracy_score(y, y_pred)
print(f"Accuracy: {acc:.4f}")

# Precision
prec = precision_score(y, y_pred)
print(f"Precision: {prec:.4f}")

# Recall
rec = recall_score(y, y_pred)
print(f"Recall: {rec:.4f}")

# F1-Score
f1 = f1_score(y, y_pred)
print(f"F1-Score: {f1:.4f}")

# Reporte completo
print("\n=== Classification Report ===\n")
print(classification_report(y, y_pred, target_names=["No Cancelado", "Cancelado"]))

### 5.3 Curva ROC y AUC

In [ ]:
# Calcular curva ROC
fpr, tpr, thresholds = roc_curve(y, y_prob)
auc = roc_auc_score(y, y_prob)

# Visualizar
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc).plot(ax=ax)
plt.plot([0, 1], [0, 1], "k--", label="Aleatorio")
plt.title("Curva ROC")
plt.legend()
plt.show()

print(f"\nAUC-ROC: {auc:.4f}")

**Pregunta:** ¿Cómo interpretas el valor de AUC obtenido? ¿Es un buen modelo?

**Pregunta:** Si el hotel quiere maximizar la detección de cancelaciones (no perder ninguna), ¿qué umbral debería usar? ¿Cuál es el trade-off?